To answer the 4th sub-question (What is the influence of humans and LLMs working together on the results of the frame annotation task of the ChangeMyView dataset?), a user study is done. The results of the study will be evaluated in this notebook.

To run this notebook, the following files need to be imported:


*   posts_forms.xlsx
*   groupA_results.xlsx
*   groupB_results.xlsx

After the files are imported, the notebook should be able to run normally.





In [14]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from nltk.metrics import agreement
from nltk.metrics.distance import jaccard_distance, masi_distance

In [15]:
# Define all 15 valid frames (exactly as they appear before the "=" in the form)
VALID_FRAMES = [
    "economic", "capacity and resources", "morality", "fairness and equality",
    "legality, constitutionality and jurisprudence", "policy prescription and evaluation",
    "crime and punishment", "security and defense", "health and safety",
    "quality of life", "cultural identity", "public opinion",
    "political", "external regulation and reputation", "other"
]

In [16]:
# Load the data
df_posts = pd.read_excel('posts_forms.xlsx')
df_a = pd.read_excel('groupA_results.xlsx')
df_b = pd.read_excel('groupB_results.xlsx')

In [17]:
# Clean up any hidden leading/trailing whitespaces in column headers
df_posts.columns = df_posts.columns.str.strip()
df_a.columns = df_a.columns.str.strip()
df_b.columns = df_b.columns.str.strip()

In [18]:
# Helper function to extract frame names
def extract_frames(response_string):
    if pd.isna(response_string):
        return []

    response_str = str(response_string)
    found_frames = []

    for frame in VALID_FRAMES:
        # Create a flexible regex pattern that matches the frame name case-insensitively
        # This handles special characters like commas and variations in layout perfectly
        pattern = re.compile(re.escape(frame), re.IGNORECASE)
        if pattern.search(response_str):
            found_frames.append(frame)

    return found_frames

In [19]:
# Robust helper function to parse Gold Standard and LLM suggestions
def parse_plus_separated(string):
    if pd.isna(string):
        return []
    # Split tokens by '+' and clean them up
    tokens = [f.strip().lower() for f in str(string).split('+')]
    cleaned_tokens = []
    for t in tokens:
        if t == 'legality' or 'jurisprudence' in t:
            cleaned_tokens.append('legality, constitutionality and jurisprudence')
        elif t == 'policy evaluation' or 'prescription' in t:
            cleaned_tokens.append('policy prescription and evaluation')
        elif 'punisment' in t or 'punishment' in t:
            cleaned_tokens.append('crime and punishment')
        else:
            cleaned_tokens.append(t)
    return cleaned_tokens

In [20]:
# Process the Gold Standard Dataframe
# Rename the first unnamed column to 'Q_Num'
df_posts.rename(columns={df_posts.columns[0]: 'Q_Num'}, inplace=True)
df_posts['Gold_Labels'] = df_posts['gold_frames'].apply(parse_plus_separated)
df_posts['LLM_Labels'] = df_posts['LLM_suggestion'].apply(parse_plus_separated)

In [21]:
# Function to melt group dataframes from Wide to Long
def process_group_df(df, group_name):
    # Create a dummy Participant ID based on row index
    df['Participant_ID'] = [f"{group_name}_{i+1}" for i in range(len(df))]

    # Keep only Comment columns and Participant_ID
    comment_cols = [c for c in df.columns if 'Comment' in c]
    df_filtered = df[['Participant_ID'] + comment_cols]

    # Melt to long format
    df_long = df_filtered.melt(id_vars=['Participant_ID'], var_name='Col_Name', value_name='Raw_Response')

    # Extract Question Number using regex (e.g., "Comment 21:" -> 21)
    df_long['Q_Num'] = df_long['Col_Name'].apply(lambda x: int(re.search(r'Comment (\d+)', x).group(1)))

    # Determine Part A or Part B based on Question Number
    df_long['Part'] = df_long['Q_Num'].apply(lambda x: 'A' if x <= 20 else 'B')

    # Extract Human Labels
    df_long['Human_Labels'] = df_long['Raw_Response'].apply(extract_frames)
    df_long['Group'] = group_name

    return df_long[['Participant_ID', 'Group', 'Part', 'Q_Num', 'Human_Labels']]

In [22]:
# Process both groups and combine
df_a_long = process_group_df(df_a, 'A')
df_b_long = process_group_df(df_b, 'B')
df_combined = pd.concat([df_a_long, df_b_long])

In [23]:
# Merge with Gold Standard to create the Master DataFrame
master_df = df_combined.merge(
    df_posts[['Q_Num', 'Gold_Labels', 'LLM_Labels', 'correct/not correct']],
    on='Q_Num',
    how='left'
)
master_df.rename(columns={'correct/not correct': 'LLM_Is_Correct'}, inplace=True)

In [24]:
master_df.head()

,Participant_ID,Group,Part,Q_Num,Human_Labels,Gold_Labels,LLM_Labels,LLM_Is_Correct
0,A_1,A,A,1,[fairness and equality],"[morality, fairness and equality]",[],NaN
1,A_2,A,A,1,[morality],"[morality, fairness and equality]",[],NaN
2,A_3,A,A,1,[other],"[morality, fairness and equality]",[],NaN
3,A_4,A,A,1,[morality],"[morality, fairness and equality]",[],NaN
4,A_5,A,A,1,"[capacity and resources, fairness and equality]","[morality, fairness and equality]",[],NaN


Standard Performance (Accuracy/recall/precision/f1-score)

In [25]:
# Initialize MultiLabelBinarizer with all valid classes
mlb = MultiLabelBinarizer(classes=VALID_FRAMES)

def calculate_metrics(df_subset):
    if df_subset.empty:
        return {"Accuracy": 0.0, 'F1 (Macro)': 0.0, 'Precision': 0.0, 'Recall': 0.0}
    y_true = mlb.fit_transform(df_subset['Gold_Labels'])
    y_pred = mlb.transform(df_subset['Human_Labels'])

    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    return {"Accuracy": round(accuracy, 4), 'F1 (Macro)': round(f1, 4), 'Precision': round(precision, 4), 'Recall': round(recall, 4)}

In [26]:
# Overall Part A vs Part B
print("--- Overall Performance ---")
print("Part A (No LLM):", calculate_metrics(master_df[master_df['Part'] == 'A']))
print("Part B (With LLM):", calculate_metrics(master_df[master_df['Part'] == 'B']))

--- Overall Performance ---
Part A (No LLM): {'Accuracy': 0.085, 'F1 (Macro)': 0.3018, 'Precision': 0.3243, 'Recall': 0.3142}
Part B (With LLM): {'Accuracy': 0.225, 'F1 (Macro)': 0.2778, 'Precision': 0.3209, 'Recall': 0.2612}


In [27]:
# Group A vs Group B on Part A
print("\nPerformance on Part A")
print("Group A (Did Part A First):", calculate_metrics(master_df[(master_df['Part'] == 'A') & (master_df['Group'] == 'A')]))
print("Group B (Did Part B First):", calculate_metrics(master_df[(master_df['Part'] == 'A') & (master_df['Group'] == 'B')]))


Performance on Part A
Group A (Did Part A First): {'Accuracy': 0.075, 'F1 (Macro)': 0.2592, 'Precision': 0.2733, 'Recall': 0.2682}
Group B (Did Part B First): {'Accuracy': 0.095, 'F1 (Macro)': 0.3444, 'Precision': 0.4075, 'Recall': 0.3601}


In [28]:
# Group A vs Group B on Part B
print("\nPerformance on Part B")
print("Group A (Did Part A First):", calculate_metrics(master_df[(master_df['Part'] == 'B') & (master_df['Group'] == 'A')]))
print("Group B (Did Part B First):", calculate_metrics(master_df[(master_df['Part'] == 'B') & (master_df['Group'] == 'B')]))


Performance on Part B
Group A (Did Part A First): {'Accuracy': 0.26, 'F1 (Macro)': 0.2811, 'Precision': 0.3387, 'Recall': 0.2527}
Group B (Did Part B First): {'Accuracy': 0.19, 'F1 (Macro)': 0.2722, 'Precision': 0.3009, 'Recall': 0.2697}


In [29]:
def get_per_frame_f1(df_subset, column_label):
    if df_subset.empty:
        return pd.Series(0.0, index=VALID_FRAMES, name=column_label)

    # Transform labels to binary arrays
    y_true = mlb.transform(df_subset['Gold_Labels'])
    y_pred = mlb.transform(df_subset['Human_Labels'])

    # Calculate F1 score for each class individually
    per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)

    return pd.Series(per_class_f1, index=VALID_FRAMES, name=column_label)


# Overall Per-Frame Comparison between Group A and Group B
f1_overall_group_a = get_per_frame_f1(master_df[master_df['Group'] == 'A'], 'Group A (Overall)')
f1_overall_group_b = get_per_frame_f1(master_df[master_df['Group'] == 'B'], 'Group B (Overall)')

df_overall_frames = pd.concat([f1_overall_group_a, f1_overall_group_b], axis=1)
print("\nOverall Per-Frame F1 Scores")
print(df_overall_frames.round(4))

# 2. Detailed Breakdown: Comparison separated by Group and Task Condition
# This isolates manual performance vs. AI-assisted performance per group
f1_a_Part_A = get_per_frame_f1(master_df[(master_df['Group'] == 'A') & (master_df['Part'] == 'A')], 'Group A (Part A)')
f1_b_Part_A = get_per_frame_f1(master_df[(master_df['Group'] == 'B') & (master_df['Part'] == 'A')], 'Group B (Part A)')
f1_a_ai     = get_per_frame_f1(master_df[(master_df['Group'] == 'A') & (master_df['Part'] == 'B')], 'Group A (Part B)')
f1_b_ai     = get_per_frame_f1(master_df[(master_df['Group'] == 'B') & (master_df['Part'] == 'B')], 'Group B (Part B)')

df_detailed_frames = pd.concat([f1_a_Part_A, f1_b_Part_A, f1_a_ai, f1_b_ai], axis=1)
print("\nPer-Frame F1 Scores by Condition")
print(df_detailed_frames.round(4))


Overall Per-Frame F1 Scores
                                               Group A (Overall)  \
economic                                                  0.6667   
capacity and resources                                    0.0000   
morality                                                  0.5285   
fairness and equality                                     0.5210   
legality, constitutionality and jurisprudence             0.6022   
policy prescription and evaluation                        0.1370   
crime and punishment                                      0.2647   
security and defense                                      0.0000   
health and safety                                         0.1290   
quality of life                                           0.4950   
cultural identity                                         0.2222   
public opinion                                            0.2174   
political                                                 0.4865   
external regulation

Inter-Annotator Agreement

In [30]:
def calculate_iaa_jaccard(df_subset):
    # Create pairs of all annotations for the same question
    distances = []
    questions = df_subset['Q_Num'].unique()

    for q in questions:
        q_data = df_subset[df_subset['Q_Num'] == q]['Human_Labels'].tolist()

        # Compare every participant with every other participant for this question
        for i in range(len(q_data)):
            for j in range(i + 1, len(q_data)):
                set1 = set(q_data[i])
                set2 = set(q_data[j])

                # Jaccard Distance
                if not set1 and not set2:
                    dist = 0.0
                else:
                    dist = jaccard_distance(set1, set2)
                distances.append(dist)

    # Return average agreement
    avg_distance = np.mean(distances) if distances else 0
    return 1 - avg_distance

In [31]:
print("Inter-Annotator Agreement")
# Overall IAA
print("Overall IAA Part A: {:.4f}".format(calculate_iaa_jaccard(master_df[master_df['Part'] == 'A'])))
print("Overall IAA Part B: {:.4f}".format(calculate_iaa_jaccard(master_df[master_df['Part'] == 'B'])))

Inter-Annotator Agreement
Overall IAA Part A: 0.2436
Overall IAA Part B: 0.3689


In [32]:
print("\nIAA Broken Down by Group")
# Group A (Order: Part A -> Part B)
print("Group A - Part A (First Task): {:.4f}".format(calculate_iaa_jaccard(master_df[(master_df['Part'] == 'A') & (master_df['Group'] == 'A')])))
print("Group A - Part B (Second Task): {:.4f}".format(calculate_iaa_jaccard(master_df[(master_df['Part'] == 'B') & (master_df['Group'] == 'A')])))

# Group B (Order: Part B -> Part A)
print("Group B - Part B (First Task): {:.4f}".format(calculate_iaa_jaccard(master_df[(master_df['Part'] == 'B') & (master_df['Group'] == 'B')])))
print("Group B - Part A (Second Task): {:.4f}".format(calculate_iaa_jaccard(master_df[(master_df['Part'] == 'A') & (master_df['Group'] == 'B')])))


IAA Broken Down by Group
Group A - Part A (First Task): 0.2294
Group A - Part B (Second Task): 0.4320
Group B - Part B (First Task): 0.3200
Group B - Part A (Second Task): 0.2653


In [33]:
def calculate_iaa_masi(df_subset):
    distances = []
    questions = df_subset['Q_Num'].unique()

    for q in questions:
        q_data = df_subset[df_subset['Q_Num'] == q]['Human_Labels'].tolist()

        # Compare every participant pair for this question
        for i in range(len(q_data)):
            for j in range(i + 1, len(q_data)):
                set1 = set(q_data[i])
                set2 = set(q_data[j])

                # MASI Distance calculation
                if not set1 and not set2:
                    dist = 0.0
                else:
                    dist = masi_distance(set1, set2)
                distances.append(dist)

    # Return average agreement
    avg_distance = np.mean(distances) if distances else 0
    return 1 - avg_distance

In [34]:
print("Inter-Annotator Agreement using MASI")
print("Overall MASI Part A: {:.4f}".format(calculate_iaa_masi(master_df[master_df['Part'] == 'A'])))
print("Overall MASI Part B: {:.4f}".format(calculate_iaa_masi(master_df[master_df['Part'] == 'B'])))

Inter-Annotator Agreement using MASI
Overall MASI Part A: 0.1849
Overall MASI Part B: 0.3112


In [35]:
print("\nMASI Broken Down by Group")
print("Group A - Part A (First Task): {:.4f}".format(calculate_iaa_masi(master_df[(master_df['Part'] == 'A') & (master_df['Group'] == 'A')])))
print("Group A - Part B (Second Task): {:.4f}".format(calculate_iaa_masi(master_df[(master_df['Part'] == 'B') & (master_df['Group'] == 'A')])))
print("Group B - Part B (First Task): {:.4f}".format(calculate_iaa_masi(master_df[(master_df['Part'] == 'B') & (master_df['Group'] == 'B')])))
print("Group B - Part A (Second Task): {:.4f}".format(calculate_iaa_masi(master_df[(master_df['Part'] == 'A') & (master_df['Group'] == 'B')])))


MASI Broken Down by Group
Group A - Part A (First Task): 0.1706
Group A - Part B (Second Task): 0.3898
Group B - Part B (First Task): 0.2470
Group B - Part A (Second Task): 0.2040


automation bias

In [36]:
# Filter to only Part B (where LLM was present)
df_part_b = master_df[master_df['Part'] == 'B'].copy()

# Function to check if the human exactly matched the LLM's suggestion
def exact_match(row):
    return set(row['Human_Labels']) == set(row['LLM_Labels'])

df_part_b['Copied_LLM'] = df_part_b.apply(exact_match, axis=1)

# Helper function to analyze bias for any given subset
def analyze_automation_bias(df_subset, label):
    correct_LMM = df_subset[df_subset['LLM_Is_Correct'] == 'correct']
    incorrect_LMM = df_subset[df_subset['LLM_Is_Correct'] == 'incorrect']

    # Calculate exact copy rates
    copy_correct = correct_LMM['Copied_LLM'].mean() * 100
    copy_incorrect = incorrect_LMM['Copied_LLM'].mean() * 100

    # Calculate actual F1 performance
    f1_correct = calculate_metrics(correct_LMM)['F1 (Macro)']
    f1_incorrect = calculate_metrics(incorrect_LMM)['F1 (Macro)']

    print(f"\n{label}")
    print(f"Exact Copy Rate (LLM was CORRECT):   {copy_correct:.2f}%")
    print(f"Exact Copy Rate (LLM was INCORRECT): {copy_incorrect:.2f}%")
    print(f"Human F1 Score (Good LLM suggestion): {f1_correct:.4f}")
    print(f"Human F1 Score (Bad LLM suggestion):  {f1_incorrect:.4f}")


# 1. Overall Bias
analyze_automation_bias(df_part_b, "OVERALL AUTOMATION BIAS")

# 2. Group A Bias
analyze_automation_bias(df_part_b[df_part_b['Group'] == 'A'], "GROUP A (Part A first, Part B second)")

# 3. Group B Bias
analyze_automation_bias(df_part_b[df_part_b['Group'] == 'B'], "GROUP B (Part B first, Part A second)")


OVERALL AUTOMATION BIAS
Exact Copy Rate (LLM was CORRECT):   30.50%
Exact Copy Rate (LLM was INCORRECT): 17.00%
Human F1 Score (Good LLM suggestion): 0.2896
Human F1 Score (Bad LLM suggestion):  0.1872

GROUP A (Part A first, Part B second)
Exact Copy Rate (LLM was CORRECT):   35.00%
Exact Copy Rate (LLM was INCORRECT): 21.00%
Human F1 Score (Good LLM suggestion): 0.2979
Human F1 Score (Bad LLM suggestion):  0.1903

GROUP B (Part B first, Part A second)
Exact Copy Rate (LLM was CORRECT):   26.00%
Exact Copy Rate (LLM was INCORRECT): 13.00%
Human F1 Score (Good LLM suggestion): 0.2795
Human F1 Score (Bad LLM suggestion):  0.1826
